# bioseq bench `report.json` viewer

Self-contained, single-file notebook for inspecting a `report.json`
produced by `./build.sh bench`. The bench writes the file to
`build/report.json` (or whatever path was passed via the
`BIOSEQ_BENCH_REPORT_PATH` environment variable). Override the
`REPORT_PATH` constant in the next cell to point somewhere else.

## Reading order

Charts are rendered in 7 tiers, top-to-bottom:

0. **Largest-bucket head-to-head** (`headline_largest_*`) — the
   single most important section for the V4 two-stack iteration.
   Compares bioseq vs minimap2 stage-by-stage (Minimizer
   generation, Chaining, Alignment, ...) on the largest sequence
   bucket the bench produced. Includes a per-stage wall-clock bar
   chart (fastest bioseq vs minimap2), a stage × algorithm
   colour-coded comparison table (green=fast, red=slow per row),
   and PMU-counter bar charts (cycles, instructions, branches,
   branch-misses, branch-miss rate) so the reader can see whether
   a stage is paying more CPU work, more branch noise, or both.
1. **Headline** (`headline_*`) — the four charts that answer 'how does
   bioseq compare to minimap2?' at a glance. Two minimap2 baselines and
   the two bioseq KSW2 + scalar-mm variants whose chaining matches
   minimap2's. Read these next.
2. **Minimizer k/w sweep** (`kmer_window_*`) — accuracy and latency
   heatmaps showing how (k, w) parameter choices affect both
   minimap2 and our port. Sourced from `run_kw_sweep` in
   `matrix_runner.cppm`.
3. **Full algorithm grid** — the same accuracy / speed / Pareto views
   over the complete 14-algorithm matrix. Use to drill into any
   variant the headline charts skip.
4. **Latency distributions** — P50/P99, CDFs, box plots, tail
   amplification. Use when sizing latency budgets.
5. **Pipeline phase breakdown** (`phase_breakdown_*`) — per-phase
   wall-clock decomposition (sketch/seed/chain/align/mapq), one
   stacked-bar chart per length bucket. Use to find which phase
   dominates wall-clock for long sequences (the canonical "why is
   bioseq slow on Mbp inputs?" view).
6. **Quality diagnostics** — position-offset histograms, MAPQ
   distribution and calibration, the per-algorithm summary table.

No external dependencies beyond `plotly` and the Python standard
library. To run:

1. Open in Jupyter (`jupyter notebook view_report.ipynb`) or VS Code.
2. Execute the cells top-to-bottom. The last cell renders every
   chart in the report grouped by the seven tiers above.

Schema: `bioseq-bench-report-v1` — see
`main/bioseq/RESEARCH/BENCHMARKING_SPEEDUP/REPORT_PLOTLY_JSON_DATA.md`
for what each chart means.

In [7]:
!pip3 install plotly nbformat


[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [8]:
# ----- configuration -----
# Either an absolute path or a path relative to the notebook directory.
# The bench writes to ./report.json by default; the build script invokes
# the binary from main/build/, so the canonical location is
# main/build/report.json.
REPORT_PATH = '../../../build/report.json'

import json
import os
import sys

import plotly.io
from IPython.display import display, Markdown

with open(REPORT_PATH, 'r') as report_file:
    report = json.load(report_file)

expected_schema = 'bioseq-bench-report-v1'
actual_schema = report.get('schema', '<missing>')
if actual_schema != expected_schema:
    print(
        f'WARNING: schema mismatch — expected {expected_schema!r}, '
        f'got {actual_schema!r}. The notebook may render incorrectly.',
        file=sys.stderr,
    )

print(f'Loaded {os.path.abspath(REPORT_PATH)}')
print(f'  schema:           {actual_schema}')
print(f'  sample_count:     {report.get("sample_count", "?")}')
print(f'  algorithm_count:  {report.get("algorithm_count", "?")}')
print(f'  chart_count:      {len(report.get("charts", {}))}')

Loaded /Users/styczynski/Code/bioseqdb/main/build/report.json
  schema:           bioseq-bench-report-v1
  sample_count:     2040
  algorithm_count:  45
  chart_count:      36


In [9]:
# Render charts grouped by tier so the headline answer comes first
# and the deep-dive details follow. Each tier prints a markdown
# section header before its charts; charts within a tier render in
# the order they appear in the JSON (which is the order
# `chart_builder::emit_report` writes them).
#
# Tiers:
#   0. Largest-bucket head-to-head (V4 priority): bioseq vs minimap2
#      stage-by-stage on the largest sequence bucket. Wall-clock bar
#      chart + colour-coded comparison table + PMU-counter charts.
#      The `LARGEST_BUCKET_CHART_ORDER` list pins the rendering order
#      so the reader sees the bar chart and the table first.
#   1. Headline: 'how does bioseq compare to minimap2?' answers (chart IDs
#      starting with `headline_`).
#   2. k/w sweep: how do minimizer length and window size affect quality
#      (chart IDs starting with `kmer_window_`).
#   3. Algorithm comparison: full 14-algorithm grid views.
#   4. Latency distributions: tail-shape charts for capacity planning.
#   5. Pipeline phase breakdown: where wall-clock goes inside one call.
#   6. Quality diagnostics: position offsets, MAPQ, summary table.
#
# Charts not assigned to a tier are listed at the end under 'Other'.

charts = report.get('charts', {})

# Tier 0 ordering — bar chart first, table second, then PMU counter
# charts in 'most useful for V4 diagnosis' order. Charts not in this
# list but matching the `headline_largest_` prefix still render in
# Tier 0 (sorted alphabetically) so a future addition shows up
# automatically.
LARGEST_BUCKET_CHART_ORDER = [
    'headline_largest_per_stage_time',
    'headline_largest_stage_table',
    'headline_largest_per_stage_branch_misses',
    'headline_largest_per_stage_branch_miss_rate',
    'headline_largest_per_stage_branches',
    'headline_largest_per_stage_instructions',
    'headline_largest_per_stage_cycles',
]


def tier0_predicate(cid):
    return cid.startswith('headline_largest_')


# (tier_label, predicate) — first-match-wins, so order matters here.
tier_specs = [
    ('Tier 0 — Largest-bucket head-to-head (bioseq vs. minimap2)',
     tier0_predicate),
    ('Tier 1 — Headline (bioseq vs. minimap2)',
     lambda cid: cid.startswith('headline_') and not cid.startswith('headline_largest_')),
    ('Tier 2 — Minimizer k/w sweep',
     lambda cid: cid.startswith('kmer_window_')),
    ('Tier 3 — Full algorithm grid',
     lambda cid: cid in {
        'seq_length_vs_speed_mean',
        'seq_length_vs_accuracy',
        'error_rate_vs_accuracy',
        'accuracy_heatmap_algorithm_vs_length',
        'accuracy_heatmap_algorithm_vs_length_error',
        'seq_length_vs_throughput',
        'speedup_vs_minimap2_reference_dp',
        'pareto_accuracy_vs_latency',
     }),
    ('Tier 4 — Latency distributions and tail behaviour',
     lambda cid: cid in {
        'seq_length_vs_speed_p99',
        'latency_distribution_box',
        'p99_over_mean_tail_amplification',
        'latency_cdf_per_algorithm',
        'seq_length_vs_memory',
     }),
    ('Tier 5 — Pipeline phase breakdown (where time goes)',
     lambda cid: cid.startswith('phase_breakdown_')),
    ('Tier 6 — Quality diagnostics',
     lambda cid: cid in {
        'position_offset_histogram',
        'mapq_distribution',
        'mapq_calibration',
        'algorithm_summary_table',
     }),
]


def order_within_tier(tier_label, ids):
    """Tier 0 has an explicit ordering; other tiers preserve report
    insertion order (which is itself stable across runs because the
    chart builder writes charts in a fixed sequence)."""
    if tier_label.startswith('Tier 0'):
        seen = set()
        ordered = []
        for cid in LARGEST_BUCKET_CHART_ORDER:
            if cid in ids and cid not in seen:
                ordered.append(cid)
                seen.add(cid)
        for cid in sorted(ids):
            if cid not in seen:
                ordered.append(cid)
                seen.add(cid)
        return ordered
    return ids


assigned = set()

for tier_label, predicate in tier_specs:
    matching_ids = [cid for cid in charts if cid not in assigned and predicate(cid)]
    matching_ids = order_within_tier(tier_label, matching_ids)
    if not matching_ids:
        continue
    display(Markdown(f'# {tier_label}'))
    for cid in matching_ids:
        chart = charts[cid]
        short_description = chart.get('short_description', cid)
        long_description = chart.get('long_description', '')
        plotly_payload = chart.get('plotly')
        if plotly_payload is None:
            display(Markdown(f'### `{cid}` — _no plotly payload, skipped_'))
            assigned.add(cid)
            continue
        display(Markdown(f'## `{cid}` — {short_description}'))
        try:
            figure = plotly.io.from_json(json.dumps(plotly_payload))
        except Exception as exc:  # noqa: BLE001
            display(Markdown(f'**plotly.io.from_json failed:** `{exc}`'))
            assigned.add(cid)
            continue
        figure.show()
        if long_description:
            display(Markdown(f'> {long_description}'))
        assigned.add(cid)

leftover_ids = [cid for cid in charts if cid not in assigned]
if leftover_ids:
    display(Markdown('# Other charts (not categorised)'))
    for cid in leftover_ids:
        chart = charts[cid]
        short_description = chart.get('short_description', cid)
        long_description = chart.get('long_description', '')
        plotly_payload = chart.get('plotly')
        if plotly_payload is None:
            display(Markdown(f'### `{cid}` — _no plotly payload, skipped_'))
            continue
        display(Markdown(f'## `{cid}` — {short_description}'))
        try:
            figure = plotly.io.from_json(json.dumps(plotly_payload))
        except Exception as exc:  # noqa: BLE001
            display(Markdown(f'**plotly.io.from_json failed:** `{exc}`'))
            continue
        figure.show()
        if long_description:
            display(Markdown(f'> {long_description}'))

# Tier 0 — Largest-bucket head-to-head (bioseq vs. minimap2)

## `headline_largest_per_stage_time` — Headline: per-stage wall-clock comparison, fastest bioseq vs minimap2 on largest sequences

> Two grouped bars per logical stage (Minimizer generation, Chaining, Alignment, ...) — left bar is minimap2 (DP chaining), right bar is the fastest bioseq config (lowest summed wall-clock across stages). Y is mean wall-clock ms per call on the largest sequence bucket the bench ran. Use this chart to spot which stage is dragging bioseq behind minimap2 (or where bioseq has already overtaken).

## `headline_largest_stage_table` — Headline: stage × algorithm comparison table (largest sequences, color-coded green→red)

> Each row is a logical pipeline stage; each column is one algorithm. Cells contain the dominant phase name + the mean wall-clock per call (e.g. 'mm_sketch_simd: 13.45 ms'). Cell background colour is a per-row gradient — green for the fastest algorithm in that stage, red for the slowest, yellow in the middle. Rows where every algorithm reports zero are coloured grey. Use this table to spot which stages bioseq dominates and which still trail minimap2.

...

> Histogram of primary mapping qualities (MAPQ, 0-60) per algorithm. Aligners that bin most primaries at MAPQ=60 (or near it) have high confidence in their placements; aligners with mass spread across the lower buckets are uncertain about where reads belong (often the right call when the input is ambiguous). MAPQ is a Phred-scaled score — 60 corresponds to 1-in-1,000,000 expected misplacement.

## `mapq_calibration` — MAPQ calibration curve (accuracy by MAPQ bin)

> For each algorithm we bin primaries into 10-wide MAPQ bins (0-9, 10-19, ..., 50-60) and report the fraction correctly placed in each bin. A well-calibrated aligner produces a monotonic curve climbing from low accuracy at low MAPQ to ~1.0 at MAPQ=60 (since MAPQ is Phred-scaled and 60 means 1-in-1,000,000 expected misplacement). Curves that cluster at high accuracy across all MAPQs indicate MAPQ deflation (the aligner is more confident than its scores reflect); curves with low accuracy at high MAPQ indicate inflation.

## `algorithm_summary_table` — Per-algorithm summary table

> Tabular roll-up of every algorithm's pair count, mean / median / P99 latency in milliseconds, and overall position-accuracy fraction. Useful as a one-glance sanity check before diving into the per-chart detail; the row ordering matches the chart legend order so the reader can cross-reference.